# Notebook d'exploration — Assistant RAG
Objectif : comprendre le pipeline RAG étape par étape, AVANT de le
retrouver organisé en fonctions réutilisables dans app/rag.py.

In [5]:
# --- 1. Imports ---
from pathlib import Path
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langchain_ollama import OllamaEmbeddings, ChatOllama

In [ ]:
# --- 2. Charger le PDF de l'entreprise ---
PDF_PATH = "data/mon_document.pdf"  # <-- à adapter

loader = PyPDFLoader(PDF_PATH)
pages = loader.load()  # une "Document" LangChain par page du PDF

print(f"Nombre de pages chargées : {len(pages)}")
print("--- Aperçu de la première page ---")
print(pages[0].page_content[:500])

Nombre de pages chargées : 7
--- Aperçu de la première page ---
ApexScale T echnologies– Dossier Institutionnel Complet Septembre 2026
]
1


In [8]:
# --- 3. Découper le texte en morceaux ("chunking") ---
# Pourquoi découper ? Un LLM ne peut pas "lire" tout un document d'un coup
# de façon fiable, et on veut pouvoir retrouver UNIQUEMENT le passage
# pertinent pour une question donnée. On coupe donc le texte en petits
# blocs ("chunks") que l'on pourra chercher individuellement.
splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,     # ~1000 caractères par morceau
    chunk_overlap=150,   # chevauchement pour ne pas couper une idée en deux
)
chunks = splitter.split_documents(pages)

print(f"Nombre de chunks créés : {len(chunks)}")
print("--- Exemple de chunk ---")
print(chunks[0].page_content)
print(chunks[0].metadata)  # contient la page d'origine -> utile pour citer les sources

Nombre de chunks créés : 11
--- Exemple de chunk ---
ApexScale T echnologies– Dossier Institutionnel Complet Septembre 2026
]
1
{'producer': 'pdfTeX-1.40.29', 'creator': 'TeX', 'creationdate': '2026-09-24T07:32:55+00:00', 'moddate': '2026-09-24T07:32:55+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.29 (TeX Live 2026) kpathsea version 6.4.2', 'trapped': '/False', 'source': 'data/mon_document.pdf', 'total_pages': 7, 'page': 0, 'page_label': '1'}


In [9]:
# --- 4. Créer les embeddings (vecteurs numériques) ---
# Un embedding transforme un texte en une liste de nombres qui capture son
# "sens". Deux textes proches en signification auront des vecteurs proches.
# On utilise "nomic-embed-text", servi localement par Ollama :
# gratuit, tourne sur CPU, léger (~270 Mo en RAM).
embeddings = OllamaEmbeddings(model="nomic-embed-text")

# Test rapide : embedder une phrase
test_vector = embeddings.embed_query("Quelle est la politique de congés ?")
print(f"Longueur du vecteur généré : {len(test_vector)}")

Longueur du vecteur généré : 768


In [10]:
# --- 5. Stocker les vecteurs dans une base vectorielle (Chroma) ---
# Chroma est une base de données spécialisée dans la recherche par
# similarité de vecteurs. Elle tourne en local, sans serveur externe,
# et sauvegarde les données sur disque via persist_directory.
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory="chroma_db_notebook",  # dossier séparé de la prod, pour expérimenter
)
print("Base vectorielle créée et sauvegardée sur disque.")

Base vectorielle créée et sauvegardée sur disque.


In [11]:
# --- 6. Tester la recherche (retrieval) ---
# On cherche les chunks les plus proches sémantiquement de la question.
question = "Quelle est la politique de congés de l'entreprise ?"
resultats = vectorstore.similarity_search(question, k=3)

for i, doc in enumerate(resultats, 1):
    print(f"\n--- Résultat {i} (page {doc.metadata.get('page')}) ---")
    print(doc.page_content[:300])


--- Résultat 1 (page 4) ---
2. Gouvernance et Équipe Dirigeante
Structure de Gouvernance
ApexScale Technologies s’appuie sur une structure de gouvernance agile, favorisant une prise
de décision rapide et alignée sur l’évolution rapide du secteur technologique mondial. Le Conseil
d’Administration supervise l’orientation stratég

--- Résultat 2 (page 4) ---
chitecture logicielle.
Sarah K., Ph.D. – Directrice Technique (CTO) & Co-fondatrice
Experte en architectures cloud et en apprentissage automatique appliqué aux infrastructures
critiques, elle est responsable de l’infrastructure technique, de la sécurité des données et de
l’industrialisation des modè

--- Résultat 3 (page 6) ---
4. Feuille de Route et Objectifs (2026-2030)
Plan Stratégique à Court et Moyen Terme
La trajectoire de croissance d’ApexScale Technologies repose sur trois axes majeurs pour les
prochaines années :
Période Objectif Stratégique Principal Indicateur Clé
2026 (S2) Lancementcommercialdelasuitelogicielle


In [12]:
# --- 7. Générer une réponse avec le LLM local ---
# ChatOllama envoie un prompt au modèle qui tourne localement via Ollama.
# "llama3.2:3b" est un bon compromis qualité / vitesse pour 16 Go de RAM
# (alternative possible : "qwen2.5:3b-instruct" ou "phi3.5:3.8b").
llm = ChatOllama(model="llama3.2:3b", temperature=0.2)

contexte = "\n\n".join(doc.page_content for doc in resultats)
prompt = f"""Réponds à la question en te basant uniquement sur ce contexte :

{contexte}

Question : {question}
Réponse :"""

reponse = llm.invoke(prompt)
print(reponse.content)

Je n'ai pas trouvé d'informations sur la politique de congés de l'entreprise ApexScale Technologies dans le dossier institutionnel fourni. Il est possible que cette information ne soit pas disponible ou qu'elle ne soit pas mentionnée dans le document.
